In [1]:
from openpyxl import load_workbook
import pandas as pd

In [2]:
file="raw-dataset.ods"

In [3]:
excel_file = pd.ExcelFile(file)

In [4]:
sheet_names =excel_file.sheet_names

In [5]:
print(sheet_names)

['qa-ทั่วไป', 'qa-research', 'ica-yesno', 'qa-sign', 'qa-creativity', 'qa-quiz', 'qa-unknow', 'note', 'qa-ขนม', 'qa-gov', 'qa-website', 'qa-g_all', 'qa-g3', 'qa-g4', 'qa-g6', 'qa-nonthai', 'qa-qwen3-body', 'qa-isan', 'qa-lanna', 'qa-central', 'qa-south', 'qa-book', 'qa-g5', 'qa-ex', 'qa-gg', 'ica-fable', 'qa-city-thai', 'ica-write-news', 'qa-exam', 'qa-qwen3', 'qa-g2', 'm-chat-qwen3', 'mt', 'ica-ภาษาถิ่น', 'qa-hack', 'onet-thai', 'r-ร่างกาย', 'r-บุคคล', 'r-กริยา', 'r-เครื่องใช้', 'r-อาหาร', 'ica-newupdate', 'qa-newupdate', 'qa-p2', 'qwen3-think', 'qa-deepseek', 'qa-ohno', 'thnking', 'qa-poetry', 'qa-thinkingonly', 'qa-พระพุทธ', 'qa-qwen2.5-14b-instruct', 'sum', 'qa-govbook', 'qa-nemotron-4-340b-instruct', 'nemotron-4-340b-instruct-m-chat', 'roplay', 'qa-nemotron-4-340b-instruct2', 'wip', 'qa-thai_food', 'multitune', 'stroy', 'ica-textbook', 'ica-task', 'qa-s', 'qa-law', 'mrc', 'qa-wangchanglm', 'qa-mumupetguide', 'qa-recoftc', 'qa-ศาสนาความเชื่อ', 'qa-coding', 'qa-Creative Writing', 'q

In [6]:
def clean_txt(t):
    return t.strip().replace("\xa0","").replace("imagine ว่าคุณ","จินตนาการว่าคุณ")

In [7]:
def get_qa():
    q=[]
    a=[]
    df= pd.read_excel(excel_file, "qa-law")
    for q1,a1,l in zip(df["q"],df["a"],df["law"]):
        if not isinstance(q1, str) or not isinstance(a1, str):
            continue
        if len(str(l))>10 and isinstance(l, str):
            q.append(clean_txt(q1+"\n\n"+l))
        else:
            q.append(clean_txt(q1))
        a.append(clean_txt(a1))
    for i in [i for i in sheet_names if i.startswith("qa-") if i!="qa-law"]:
        df= pd.read_excel(excel_file, i)
        # print(i)
        if "r" in df.keys():
            for q1,a1,r1 in zip(df["q"],df["a"],df["r"]):
                if not isinstance(q1, str) or not isinstance(a1, str):
                    continue
                q.append(clean_txt(q1))
                if isinstance(r1,str):
                    a.append(clean_txt(a1+" "+r1))
                else:
                    a.append(clean_txt(a1))
            continue
        for q1,a1 in zip(df["q"],df["a"]):
            if isinstance(q1, str) and isinstance(a1, str):
                q.append(clean_txt(q1))
                a.append(clean_txt(a1))
    for j in [i for i in sheet_names if i.startswith("ica-")]:
        df= pd.read_excel(excel_file, j)
        for _,i in df.iterrows():
            if not isinstance(i["i"], str) or not isinstance(i["a"], str):
                continue
            if isinstance(i["c"],str):
                q.append(clean_txt(i["i"].strip()+"\n"+i["c"]))
            else:
                q.append(clean_txt(i["i"].strip()))
            
            if "r" in i.keys():
                if isinstance(i["r"],str):
                    a.append(clean_txt(i["a"]+"\n"+i["r"]))
                else:
                    a.append(clean_txt(i["a"]))
            else:
                a.append(clean_txt(i["a"]))
    return [[{"role":"user","content":i},{"role":"assistant","content":j}] for i,j in zip(q,a)]
def get_han_chat():
    df = pd.read_excel(excel_file, "m-chat")
    #df = pd.read_csv("hankub - chat.csv")
    keys = list(df.keys())
    list_all=[]
    for index, row in df.iterrows():
        list_chat=[]
        for r in keys:
            if not isinstance(row[r],str):
                break
            if r.startswith("q"):
                list_chat.append({"role":"user","content":clean_txt(row[r])})
            else:
                list_chat.append({"role":"assistant","content":clean_txt(row[r])})
        if list_chat == []:
            continue
        list_all.append(list_chat)
    return list_all
def get_han_nemo_chat():
    df = pd.read_excel(excel_file, "nemotron-4-340b-instruct-m-chat")
    #df = pd.read_csv("hankub - chat.csv")
    keys = list(df.keys())
    list_all=[]
    for index, row in df.iterrows():
        list_chat=[]
        for r in keys:
            if not isinstance(row[r],str):
                break
            if r.startswith("q"):
                list_chat.append({"role":"user","content":clean_txt(row[r])})
            else:
                list_chat.append({"role":"assistant","content":clean_txt(row[r])})
        if list_chat == []:
            continue
        list_all.append(list_chat)
    df = pd.read_excel(excel_file, "m-chat-qwen3")
    #df = pd.read_csv("hankub - chat.csv")
    keys = list(df.keys())
    for index, row in df.iterrows():
        list_chat=[]
        for r in keys:
            if not isinstance(row[r],str):
                break
            if r.startswith("q"):
                list_chat.append({"role":"user","content":clean_txt(row[r])})
            else:
                list_chat.append({"role":"assistant","content":clean_txt(row[r])})
        if list_chat == []:
            continue
        list_all.append(list_chat)
    return list_all


def get_mrc():
    df = pd.read_excel(excel_file, "mrc")
    list_all=[]
    for index, row in df.iterrows():
        list_chat=[]
        if not isinstance(row["context"], str):
            continue
        list_chat.append({"role":"user","content":clean_txt(f"จงตอบคำถามจากบทความต่อไปนี้\nบทความ: {row['context']}\nคำถาม: {row['question']}")})
        list_chat.append({"role":"assistant","content":clean_txt(row["answer"])})
        list_all.append(list_chat)
    return list_all

def get_roplay():
    df = pd.read_excel(excel_file, "roplay")
    list_all=[]
    for index, row in df.iterrows():
        list_chat=[]
        list_chat.append({"role":"system","content":clean_txt(row['system'])})
        list_chat.append({"role":"user","content":clean_txt(row['q'])})
        list_chat.append({"role":"assistant","content":clean_txt(row["a"])})
        list_all.append(list_chat)
    return list_all

def get_multitune():
    df = pd.read_excel(excel_file, "multitune")
    #df = pd.read_csv("hankub - chat.csv")
    keys = list(df.keys())
    keys.remove("system_prompt")
    list_all=[]
    for index, row in df.iterrows():
        list_chat=[]
        if not isinstance(row["system_prompt"],str):
            continue
        list_chat.append({"role":"system","content":clean_txt(row["system_prompt"])})
        for r in keys:
            if not isinstance(row[r],str):
                break
            if r.startswith("q"):
                list_chat.append({"role":"user","content":clean_txt(row[r])})
            else:
                list_chat.append({"role":"assistant","content":clean_txt(row[r])})
        if list_chat == []:
            continue
        list_all.append(list_chat)
    return list_all

def get_r():
    _d=[]
    for i in [i for i in sheet_names if i.startswith("r-")]:
        df= pd.read_excel(excel_file, i)
        for w,a in zip(list(df["w"]),list(df['a'])):
            if not isinstance(a, str):
                continue
            list_chat=[]
            list_chat.append({"role":"user","content":f"จงบอกความหมายของคําราชาศัพท์: {w}"})
            list_chat.append({"role":"assistant","content":f'ความหมายของคําราชาศัพท์ คำว่า "{w}" คือ "{a}"'})
            _d.append(list_chat)
    return _d

def get_mt():
    df = pd.read_excel(excel_file, "mt")
    list_all=[]
    for index, row in df.iterrows():
        list_chat=[]
        if not isinstance(row["src"], str):
            continue
        list_chat.append({"role":"user","content":clean_txt(f"แปลข้อความต่อไปเป็นภาษาไทย:\n{row['target']}")})
        list_chat.append({"role":"assistant","content":clean_txt(row["src"])})
        list_all.append(list_chat)
        list_chat=[]
        list_chat.append({"role":"user","content":clean_txt(f"แปลข้อความต่อไปจากภาษาไทยเป็น{row['x']}:\n{row['src']}")})
        list_chat.append({"role":"assistant","content":clean_txt(row["target"])})
        list_all.append(list_chat)
    return list_all

def get_all():
    return get_r()+get_qa()+get_roplay()+get_han_chat()+get_mrc()+get_multitune()+get_han_nemo_chat()+get_mt()

In [8]:
l=get_qa()

In [9]:
l[131]

[{'role': 'user',
  'content': 'การแจ้ง ยื่น หรือส่งหนังสือหรือเอกสารให้บุคคลใด ต้องดำเนินการอย่างไร\n\nพระราชบัญญัติว่าด้วยการปรับเป็นพินัย พ.ศ. ๒๕๖๕มาตรา ๖ ในกรณีพระราชบัญญัตินี้มิได้กำหนดไว้เป็นประการอื่น การใดที่กำหนดให้แจ้งยื่น หรือส่งหนังสือหรือเอกสารให้บุคคลใดเป็นการเฉพาะ ถ้าได้แจ้ง ยื่น หรือส่งหนังสือหรือเอกสารให้บุคคลนั้น ณ ภูมิลำเนาหรือที่อยู่ที่ปรากฏตามหลักฐานทางทะเบียนตามกฎหมายว่าด้วยการทะเบียนราษฎรให้ถือว่าได้แจ้ง ยื่น หรือส่งโดยชอบด้วยพระราชบัญญัตินี้แล้วในกรณีปรากฏต่อเจ้าหน้าที่ของรัฐว่าผู้กระทำความผิดทางพินัยไม่สามารถพูดหรือได้ยิน หรือสื่อความหมายได้ ให้แจ้งหนังสือหรือเอกสารในรูปแบบที่ผู้กระทำความผิดทางพินัยสามารถเข้าใจได้หรือจะแจ้งให้ผู้กระทำความผิดทางพินัยทราบด้วยการสื่อความหมายโดยวิธีอื่นก็ได้ในกระบวนการพิจารณาความผิดทางพินัย หากผู้ถูกกล่าวหา หรือพยานไม่สามารถพูดหรือได้ยินหรือสื่อความหมายได้ ให้เจ้าหน้าที่ของรัฐ พนักงานอัยการ หรือศาล จัดหาล่ามภาษามือให้หรือจัดให้ถาม ตอบ หรือสื่อความหมายโดยวิธีอื่นที่เห็นสมควร'},
 {'role': 'assistant',
  'content': 'โดยหลักแล้ว การแจ้

In [10]:
# l[-1]

In [11]:
list_test=get_all()

In [12]:
len(list_test) # 13139

15821

In [13]:
list_test[3443]

[{'role': 'user', 'content': 'เปรียบเทียบภาคกลางกับภาคใต้ของไทย'},
 {'role': 'assistant',
  'content': '### การเปรียบเทียบภาคกลางกับภาคใต้ของไทย#### 1. **ภูมิศาสตร์**- **ภาคกลาง**:    ส่วนใหญ่เป็นที่ราบลุ่มแม่น้ำเจ้าพระยา ดินแดนอุดมสมบูรณ์เหมาะสำหรับเกษตรกรรม เช่น การปลูกข้าว ไม่มีภูเขาสูงมาก มีแม่น้ำเจ้าพระยาไหลผ่านเป็นเส้นเลือดใหญ่ รวมถึงพื้นที่ชลประทานที่สำคัญ เช่น จังหวัดอยุธยา สุพรรณบุรี นครปฐม  - **ภาคใต้**:    มีภูมิประเทศหลากหลาย ทั้งเทือกเขา ชายฝั่งทะเลอันดามันและอ่าวไทย เกาะ (เช่น ภูเก็ต กระบี่ สุราษฎร์ธานี) และป่าฝนเขตร้อน บางพื้นที่มีภูเขาหินปูนและที่ราบสูง เช่น จังหวัดนครศรีธรรมราช สงขลา  #### 2. **เศรษฐกิจ**- **ภาคกลาง**:    เป็นศูนย์กลางเศรษฐกิจของประเทศ โดยกรุงเทพมหานครเป็น hub ด้านการเงิน การค้า การผลิต และบริการ อุตสาหกรรมหลักได้แก่ อิเล็กทรอนิกส์ ยานยนต์ อาหารแปรรูป และการท่องเที่ยวเชิงเมือง (เช่น ตลาดนัด วัดวาอาราม)  - **ภาคใต้**:    พึ่งพาการท่องเที่ยวชายหาดและเกาะ (เช่น ภูเก็ต พัทยา) การเกษตรยางพารา (ผลิตยางรายใหญ่สุดของโลก) ปาล์มน้ำมัน และการประมง แต่พื้นที่สามจั

In [14]:
import json
with open('han-instruction-dataset.json', 'w', encoding='utf8') as json_file:
    json.dump(list_test, json_file, ensure_ascii=False)

In [17]:
from datasets import Dataset
ds = Dataset.from_dict({"messages":list_test})

In [18]:
ds.push_to_hub("pythainlp/han-instruction-dataset",private=False)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/pythainlp/han-instruction-dataset/commit/e2732be1c5ae770020543ab16c88039a6e123126', commit_message='Upload dataset', commit_description='', oid='e2732be1c5ae770020543ab16c88039a6e123126', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pythainlp/han-instruction-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pythainlp/han-instruction-dataset'), pr_revision=None, pr_num=None)